# Forest Impact Simulator (Python)

Python port of the [Forest Impact Simulator](https://github.com/karimogit/Forest-Impact-Simulator) web app.

Calculations (growth curves, clear-cutting carbon, biodiversity/resilience scales, water/air percentages, planting timelines) are aligned with the original TypeScript implementation.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Ensure the package is importable when the notebook runs from the repo root
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import matplotlib.pyplot as plt

from forest_impact import ForestImpactSimulator, TREE_TYPES, get_tree_by_name, format_area
from forest_impact.export import write_exports, generate_csv
from forest_impact.planting import calculate_planting_timeline

simulator = ForestImpactSimulator()
print(f"Simulator v{simulator.version} ready — {len(TREE_TYPES)} tree species loaded")

## 2. Load forest plot data

Supported input columns: `plot_id`, `area` (ha), `tree_type`, `tree_count`, `tree_density`, `latitude`, `longitude`.

You can also load the comprehensive one-row export CSV (`sample-forest-planting-data.csv`).

In [ ]:
def _num(value, default=None):
    if value is None or (isinstance(value, str) and value.strip() == ""):
        return default
    try:
        if pd.isna(value):
            return default
    except TypeError:
        pass
    return float(value)


def load_plot_csv(path: str):
    df = pd.read_csv(path)
    # Comprehensive single-row export format
    if "area_hectares" in df.columns and "total_trees" in df.columns:
        row = df.iloc[0]
        names = [part.strip() for part in str(row.get("tree_names") or "Oak").split(";") if part.strip() and part.strip().lower() != "nan"]
        if not names:
            names = ["Oak"]
        raw_percentages = row.get("tree_percentages")
        percentages = None
        if pd.notna(raw_percentages):
            parts = [part.strip() for part in str(raw_percentages).split(";") if part.strip()]
            if len(parts) == len(names):
                values = [float(part) for part in parts]
                if abs(sum(values) - 100) < 0.01:
                    percentages = {name: value for name, value in zip(names, values)}

        region = {
            "north": _num(row.get("region_north")),
            "south": _num(row.get("region_south")),
            "east": _num(row.get("region_east")),
            "west": _num(row.get("region_west")),
        }
        if any(value is None for value in region.values()):
            region = None

        meta = {
            "source": "comprehensive",
            "years": int(_num(row.get("simulation_years"), 50)),
            "temperature": _num(row.get("temperature_c")),
            "precipitation": _num(row.get("precipitation_mm")),
            "soil_carbon": _num(row.get("soil_carbon_g_kg")),
            "soil_ph": _num(row.get("soil_ph")),
            "soil_texture": None if pd.isna(row.get("soil_texture")) else str(row.get("soil_texture")),
            "spacing": _num(row.get("spacing_meters")),
            "region": region,
            "trees": names,
            "total_trees": int(_num(row.get("total_trees"), 0)),
            "area": _num(row.get("area_hectares"), 0.0),
            "latitude": _num(row.get("latitude")),
            "longitude": _num(row.get("longitude")),
            "percentages": percentages,
        }
        plots = [{
            "plot_id": "Imported",
            "area": meta["area"],
            "tree_type": names[0],
            "tree_count": meta["total_trees"],
            "latitude": meta["latitude"],
            "longitude": meta["longitude"],
        }]
        return plots, meta

    plots = []
    for i, row in df.iterrows():
        plots.append({
            "plot_id": str(row.get("plot_id", f"Plot_{i+1:03d}")),
            "area": _num(row.get("area"), 0.0) or 0.0,
            "tree_type": "Oak" if pd.isna(row.get("tree_type")) else str(row.get("tree_type")),
            "tree_count": int(_num(row.get("tree_count"), 0) or 0),
            "tree_density": _num(row.get("tree_density"), 0.0) or 0.0,
            "latitude": _num(row["latitude"]) if "latitude" in df.columns else None,
            "longitude": _num(row["longitude"]) if "longitude" in df.columns else None,
        })
    return plots, {"years": 50, "source": "plots"}


plots_data, import_meta = load_plot_csv("sample-forest-planting-data.csv")
display(pd.DataFrame(plots_data))
print("Import meta:", {k: v for k, v in import_meta.items() if k != "region"})


## 3. Configure and run simulation

Set `mode` to `planting` or `clear-cutting`. For clear-cutting, set `average_tree_age`.

In [ ]:
MODE = "planting"          # or "clear-cutting"
YEARS = import_meta.get("years", 50)
AVERAGE_TREE_AGE = 20      # used in clear-cutting mode

if import_meta.get("source") == "comprehensive":
    results = simulator.simulate(
        trees=import_meta["trees"],
        total_trees=import_meta["total_trees"],
        area_hectares=import_meta["area"],
        years=YEARS,
        mode=MODE,
        latitude=import_meta.get("latitude"),
        longitude=import_meta.get("longitude"),
        region=import_meta.get("region"),
        temperature=import_meta.get("temperature"),
        precipitation=import_meta.get("precipitation"),
        soil_carbon=import_meta.get("soil_carbon"),
        soil_ph=import_meta.get("soil_ph"),
        soil_texture=import_meta.get("soil_texture"),
        percentages=import_meta.get("percentages"),
        spacing_meters=import_meta.get("spacing"),
        average_tree_age=AVERAGE_TREE_AGE,
    )
else:
    results = simulator.simulate_from_plots(
        plots_data,
        years=YEARS,
        mode=MODE,
        average_tree_age=AVERAGE_TREE_AGE,
        temperature=import_meta.get("temperature"),
        precipitation=import_meta.get("precipitation"),
        soil_carbon=import_meta.get("soil_carbon"),
    )

impact = results["impactResults"]
planting = results["plantingData"]

print(f"Mode: {MODE} | Years: {YEARS}")
print(f"Area: {format_area(planting['area'])} | Trees: {planting['totalTrees']:,}")
print(f"Annual carbon (period average): {impact['carbonSequestration']:,.1f} kg CO₂/year")
print(f"Total carbon over {YEARS} years: {impact['totalCarbon']:,.1f} kg CO₂")
print(f"Biodiversity: {impact['biodiversityImpact']:.1f}/5")
print(f"Biodiversity period average: {impact['averageBiodiversity']:.1f}/5")
print(f"Resilience: {impact['forestResilience']:.1f}/5")
print(f"Resilience period average: {impact['averageResilience']:.1f}/5")
print(f"Water retention: {impact['waterRetention']:.0f}%")
print(f"Air quality: {impact['airQualityImprovement']:.0f}%")
print(f"Social impact: {results['socialImpact']:.1f}/5")
print("Comparisons:", {k: round(v, 1) for k, v in results['comparisons'].items()})
if impact.get("clearCuttingBreakdown"):
    print("Clear-cutting breakdown:", impact["clearCuttingBreakdown"])


## 4. Visualize impacts

In [ ]:
labels = ["Biodiversity", "Resilience", "Water %", "Air %"]
values = [
    impact["biodiversityImpact"],
    impact["forestResilience"],
    impact["waterRetention"] / 20,  # scale 0-95% toward ~0-5 for charting
    max(0, impact["airQualityImprovement"]) / 20,
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(labels, values, color=["#2d6a4f", "#40916c", "#52b788", "#74c69d"])
axes[0].set_ylim(0, 5.5)
axes[0].set_title("Impact scores (water/air scaled ÷20)")
axes[0].set_ylabel("Score")

if MODE == "planting":
    years = list(range(1, YEARS + 1))
    from forest_impact.growth import get_planting_growth_factor
    mature = impact["matureAnnualCarbon"]
    cumulative = []
    running = 0.0
    for y in years:
        running += mature * get_planting_growth_factor(y)
        cumulative.append(running / 1000)  # metric tons
    axes[1].plot(years, cumulative, color="#1b4332", linewidth=2)
    axes[1].set_title("Cumulative carbon (growth curve)")
    axes[1].set_xlabel("Year")
    axes[1].set_ylabel("Metric tons CO₂")
    axes[1].grid(True, alpha=0.3)
else:
    breakdown = impact["clearCuttingBreakdown"]
    axes[1].bar(
        ["Immediate release", "Lost future"],
        [breakdown["immediate"] / 1000, breakdown["lost_future"] / 1000],
        color=["#9b2226", "#bb3e03"],
    )
    axes[1].set_title("Clear-cutting carbon")
    axes[1].set_ylabel("Metric tons CO₂")

plt.tight_layout()
plt.show()


## 5. Export results

Exports match the original web app formats: comprehensive CSV, JSON, and GeoJSON.

In [ ]:
from datetime import datetime, timezone

prefix = f"forest_impact_results_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
paths = write_exports(results, prefix)
print("Wrote:")
for kind, path in paths.items():
    print(f"  {kind}: {path}")

print("\nCSV preview:")
print(generate_csv(results))


## 6. Quick clear-cutting example

In [ ]:
clear_results = simulator.simulate(
    trees=["Oak"],
    total_trees=planting["totalTrees"],
    area_hectares=planting["area"],
    years=YEARS,
    mode="clear-cutting",
    latitude=results["metadata"]["location"]["latitude"],
    longitude=results["metadata"]["location"]["longitude"],
    average_tree_age=AVERAGE_TREE_AGE,
    temperature=import_meta.get("temperature"),
)
cc = clear_results["impactResults"]["clearCuttingBreakdown"]
print(f"Immediate release: {cc['immediate']:,.1f} kg CO₂")
print(f"Lost future sequestration: {cc['lost_future']:,.1f} kg CO₂")
print(f"Total emissions: {cc['total']:,.1f} kg CO₂")
print(f"Air quality impact: {clear_results['impactResults']['airQualityImprovement']:.0f}%")